# VASP: Introdução

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares) 

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

## Molécula de H2O

Criando a molécula de água (H2O) utilizando o ASE

In [ ]:
from ase import Atoms, Atom

h2omol = Atoms([Atom('O', [0, 0, 0]),
                Atom('H', [0.0, -0.760265, 0.588373]),
                Atom('H', [0.0, 0.760265, 0.588373])])

Visualizando a molécula com o ASE

In [ ]:
from ase.visualize import view

view(h2omol)

In [ ]:
view(h2omol, viewer='x3d')

### Usando o VASP com o ASE

Adicionando um vácuo de 4 Angstrom ao redor da molécula para criar a caixa de simulação com condição de contorno períodica

In [ ]:
h2omol.center(vacuum=4.0) # caixa com 4 Angstroms de vácuo 
h2omol.pbc = True # condição de contorno periódica

Importando o VASP calculator para o  ASE

In [ ]:
from ase.calculators.vasp import Vasp

calc = Vasp(xc='PBE',                                   # funcional
            encut=350,     # safe default for PAW-PBE sets
            kpts=[1, 1, 1],gamma=True,  # Somente pontos Gamma em Fourier
            ibrion=-1,       # Calcula SCF 
            directory='water' # pasta em que os cálculos serão armazenados
        )

Anexando calculadora a molécula

In [ ]:
h2omol.calc = calc

Exportando resultado da energia

In [ ]:
E0 = h2omol.get_potential_energy()       

print('Energia: {:.3f} eV'.format(E0))

### Analisando orbitais atômicos

Determinando as energias dos orbitais moleculares

In [ ]:
calc.get_eigenvalues()

Determinando as ocupações

In [ ]:
calc.get_occupation_numbers()

Determinando as energias dos orbitais HOMO e LUMO

In [ ]:
homo, lumo = calc.get_homo_lumo()

print(f"Lumo: {lumo:.3f} eV")
print(f"Homo: {homo:.3f} eV")
print('----------')
print(f"Diferença: {lumo-homo:.3f} eV")

De forma mais elegante

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

gap = (lumo - homo)

# Eigenvalues and occupations (Kohn–Sham levels)
eigs = np.array(calc.get_eigenvalues())
occ  = np.array(calc.get_occupation_numbers())

# Sort for a clean plot
order = np.argsort(eigs)
eigs, occ = eigs[order], occ[order]

# Split occupied / unoccupied
occ_mask = occ > 0.5
E_occ = eigs[occ_mask]
E_unocc = eigs[~occ_mask]

# --- plotting (single chart, no explicit colors) ---
plt.figure(figsize=(3, 3.2), dpi=140)

# Sticks for occupied and unoccupied states
for E in E_occ:
    plt.plot([0, 0.6], [E, E],color='C0')
for E in E_unocc:
    plt.plot([0.7, 1.3], [E, E],color='grey')

# Highlight HOMO / LUMO
plt.plot([0, 1.3],[homo, homo],color='red', linewidth=2)
plt.plot([0, 1.3],[lumo, lumo],color='red', linewidth=2)

# Annotate
plt.text(1.6,homo-1, 'HOMO', ha='center', va='bottom', fontsize=9)
plt.text(1.6, lumo-1, 'LUMO', ha='center', va='bottom', fontsize=9)
plt.title(f'HOMO-LUMO gap = {gap:.3f} eV')

plt.ylabel('Energia (eV)')
plt.xticks([])          # hide y-axis (pure spectrum)
plt.ylim(min(eigs)-1, max(eigs)+1)
plt.grid(True, axis='y', linestyle=':', linewidth=0.7)
plt.tight_layout()

print(f"HOMO = {homo:.3f} eV, LUMO = {lumo:.3f} eV, GAP = {gap:.3f} eV")

In [ ]:
print(f'Comprimento de onda da radiação pro gap é de {1239.8/gap:.3f} nm')

## Sólidos

Criando cristal de cobre (Cu)

In [ ]:
a = 2.53
Cu_crystal = Atoms([Atom("Cu", (0, 0, 0))],
                cell=0.5 * a * np.array([[1.0, 1.0, 0.0],
                [0.0, 1.0, 1.0],
                [1.0, 0.0, 1.0]]), pbc=True)

print(Cu_crystal)
print("Cell:", Cu_crystal.get_cell())
print("Positions:\n", Cu_crystal.get_positions())

visualizando estrutura

In [ ]:
view(Cu_crystal, viewer='x3d')

Criando calculadora do VASP

In [ ]:
calc = Vasp(directory='Cu',
                xc='PBE',
                kpts=[6, 6, 6],  # specifies k-points
                encut=350,
                atoms = Cu_crystal)

Energia da célula unitária primitiva

In [ ]:
E_Cu = Cu_crystal.get_potential_energy()

print(f'Energia total do cristal de Cu: {E_Cu:.3f} eV')

Energia por átomo da célula unitária

In [ ]:
E_per_Cu = E_Cu / len(Cu_crystal)

print(f'Energia por átomo de Cu {E_per_Cu:.3f} eV')